# Laboratorio End-to-End: CML + California Housing

Este notebook implementa un flujo completo inspirado en `chapter-5-2-cicd-estrategias-en-el-mundo-real-copy.ipynb`, adaptado a un caso ejecutable con CML.

Objetivo operativo:

1. Construir y validar datos (gate de calidad).
2. Ejecutar smoke train en CI.
3. Entrenar/evaluar varios modelos de regresion.
4. Generar reporte markdown listo para comentario CML en PR.


## 0) Prerrequisitos

- Ejecutar este notebook desde la raiz del repo `dbx-ucm-productivizacion`.
- Tener `uv` instalado.
- Tener acceso a Internet para descargar California Housing la primera vez.


In [ ]:
from __future__ import annotations

import json
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

def run(cmd: str, cwd: Path, check: bool = True) -> subprocess.CompletedProcess:
    print(f'$ {cmd}')
    completed = subprocess.run(
        cmd,
        cwd=str(cwd),
        shell=True,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout.strip())
    if completed.stderr:
        print(completed.stderr.strip())
    if check and completed.returncode != 0:
        raise RuntimeError(f'Command failed ({completed.returncode}): {cmd}')
    return completed

repo_root = Path.cwd().resolve()
if not (repo_root / 'cml_project').exists():
    raise FileNotFoundError('No se encontro cml_project. Abre el notebook en la raiz del repo.')

project_root = repo_root / 'cml_project'
artifacts_dir = project_root / 'artifacts'
print('Repo root:', repo_root)
print('Project root:', project_root)


## 1) Instalar dependencias


In [ ]:
run('uv sync --locked --package cml_project', cwd=repo_root)


## 2) Ejecutar pipeline local (equivalente a jobs CI/CT del workflow)

Orden:

1. `get_data.py`
2. `validate_data.py`
3. `smoke_train.py`
4. `train.py`
5. `build_report.py`


In [ ]:
commands = [
    'uv run --project cml_project python cml_project/src/get_data.py',
    'uv run --project cml_project python cml_project/src/validate_data.py',
    'uv run --project cml_project python cml_project/src/smoke_train.py',
    'uv run --project cml_project python cml_project/src/train.py',
    'uv run --project cml_project python cml_project/src/build_report.py',
]
for command in commands:
    run(command, cwd=repo_root)


## 3) Inspeccionar salidas de validacion y smoke gate


In [ ]:
validation = json.loads((artifacts_dir / 'data_validation.json').read_text(encoding='utf-8'))
smoke = json.loads((artifacts_dir / 'smoke_train.json').read_text(encoding='utf-8'))

display(Markdown('### Data Validation'))
display(pd.DataFrame([validation['rows']]))
display(pd.DataFrame([validation['target_stats']]))

display(Markdown('### Smoke Train'))
display(pd.DataFrame([smoke]))


## 4) Leaderboard de modelos


In [ ]:
leaderboard = pd.read_csv(artifacts_dir / 'leaderboard.csv')
display(leaderboard)

metrics = json.loads((artifacts_dir / 'metrics.json').read_text(encoding='utf-8'))
print('Best model:', metrics['best_model'])
print('Selection metric:', metrics['selection_metric'])


## 5) Artefactos visuales y reporte final


In [ ]:
residual_plot = artifacts_dir / 'residuals_plot.png'
report_path = project_root / 'report.md'

display(Image(filename=str(residual_plot)))
display(Markdown(report_path.read_text(encoding='utf-8')))


## 6) Conexion con CI/CT/CD

- CI: `get_data + validate_data + smoke_train` como puertas rapidas y deterministas.
- CT: `train.py` como entrenamiento completo y seleccion de candidato por contrato de metrica.
- CD: promocion de `best_model.joblib` fuera de este notebook, sin reentrenar en deploy.

Sugerencia de extension para alumnos: anadir un umbral de promocion (ej. `rmse <= 0.50`) y romper pipeline si no se cumple.
